# PandemicWatch-AI: Machine Learning Model Training Pipeline
## Training Multi-Modal Outbreak Risk Regressors and Disease Classifiers

Pipeline Architecture:
```
News signals
      +
Disease signals
      +
Environmental data
      +
Search trends
      ↓
Feature Processing (StandardScaler)
      ↓
ML Model (GradientBoostingRegressor / RandomForestClassifier)
      ↓
Risk Score & Pathogen Triage
```

In [1]:
import numpy as np
import os
import joblib
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

print("Scikit-learn and Joblib imported successfully.")

### 1. Construct Training Multi-Signal Feature Matrix
Features: `[news_mentions, clinical_reports, temperature_c, humidity_pct, search_trend]`

In [2]:
X_train = np.array([
    [25, 45, 28.5, 80, 92],  # High risk
    [18, 30, 27.2, 75, 78],  # High risk (Tamil Nadu profile)
    [12, 18, 26.0, 68, 65],  # Medium risk
    [8,  12, 25.0, 60, 52],  # Medium risk
    [3,   2, 22.0, 45, 25],  # Low / Normal (Karnataka baseline)
    [1,   0, 20.5, 40, 15],  # Low / Normal
    [22, 38, 29.0, 82, 88],  # High risk
    [14, 22, 27.5, 70, 62],  # Medium risk
    [2,   1, 23.0, 50, 20],  # Low
    [30, 55, 30.0, 85, 98],  # Extreme high
])

y_risk = np.array([84.0, 72.0, 58.0, 46.0, 22.0, 14.0, 80.0, 54.0, 18.0, 95.0])
print(f"X shape: {X_train.shape}, y shape: {y_risk.shape}")

### 2. Feature Processing (StandardScaler)

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
print("Mean features:", scaler.mean_)
print("Scale features:", scaler.scale_)

### 3. Train GradientBoostingRegressor for Outbreak Risk Prediction

In [4]:
risk_model = GradientBoostingRegressor(n_estimators=35, learning_rate=0.1, max_depth=3, random_state=42)
risk_model.fit(X_scaled, y_risk)

train_preds = risk_model.predict(X_scaled)
r2 = r2_score(y_risk, train_preds)
rmse = np.sqrt(mean_squared_error(y_risk, train_preds))
print(f"Training R² Score: {r2:.4f}")
print(f"Training RMSE: {rmse:.4f}")

### 4. Train RandomForestClassifier for Pathogen Triage

In [5]:
y_pathogen = np.array([0, 1, 0, 1, 2, 2, 0, 1, 2, 0])  # 0: Respiratory, 1: Dengue, 2: Baseline
disease_model = RandomForestClassifier(n_estimators=20, random_state=42)
disease_model.fit(X_scaled, y_pathogen)
print("Disease classifier trained successfully.")

### 5. Export Pickled Artifacts to `models/`

In [6]:
models_dir = os.path.abspath("../models")
os.makedirs(models_dir, exist_ok=True)

joblib.dump(risk_model, os.path.join(models_dir, "risk_model.pkl"))
joblib.dump(scaler, os.path.join(models_dir, "preprocessing.pkl"))
joblib.dump(disease_model, os.path.join(models_dir, "disease_model.pkl"))

print("Exported:")
print(f" - {models_dir}/risk_model.pkl")
print(f" - {models_dir}/preprocessing.pkl")
print(f" - {models_dir}/disease_model.pkl")